# Memory, search, market APIs, and caching: practice

Connect memory server → store/recall entities → connect search → inspect market data → cache end-of-day prices → compare local servers and hosted services.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [6_mcp/3_lab3.ipynb](../../../../../6_mcp/3_lab3.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. MCP tools, resources, and server lifetime

A tool performs an operation; a resource supplies readable context. A client discovers tool descriptions and calls them while the server is connected. This fixture is not a protocol transport.

**Predict:** Is listing tool names the same as executing them?

In [ ]:
from contextlib import contextmanager
calls = []
@contextmanager
def practice_server():
    tools = {"add": lambda a, b: a + b}
    try:
        calls.append("connected")
        yield tools
    finally:
        calls.append("closed")
with practice_server() as tools:
    print("Discovered:", list(tools))
    result = tools["add"](12, 30)
    print("Called add:", result)
assert result == 42 and calls == ["connected", "closed"]

**Check:** No. Discovery returns descriptions. A real stdio subprocess also has a working directory, environment, and protocol stream.

**Change one thing:** Add a read-only dictionary resource and a tool returning a fixed date. Explain how discovery differs from invocation.

## 2. Reducer vs. checkpoints vs. conversation identity

A reducer combines state updates. A checkpointer retains state across invocations. A thread_id selects a conversation. This uses SQLite in memory, so it creates no database file.

**Predict:** Does a different thread_id share the same list?

In [ ]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
class MemoryState(TypedDict):
    turns: Annotated[list[str], add]
builder = StateGraph(MemoryState)
builder.add_node("reply", lambda state: {"turns": ["assistant reply"]})
builder.add_edge(START, "reply")
builder.add_edge("reply", END)
with SqliteSaver.from_conn_string(":memory:") as saver:
    graph = builder.compile(checkpointer=saver)
    config = {"configurable": {"thread_id": "practice-1"}}
    graph.invoke({"turns": ["first user turn"]}, config)
    result = graph.invoke({"turns": ["second user turn"]}, config)
    separate = graph.invoke({"turns": ["new user"]}, {"configurable": {"thread_id": "practice-2"}})
    print(result["turns"], separate["turns"])
    assert len(result["turns"]) == 4 and len(separate["turns"]) == 2

**Check:** No. Same ID continues a thread; a different ID selects another. In-memory SQLite here disappears when the connection closes.

**Change one thing:** Replace :memory: with a deliberately chosen practice database only if you want persistence. Explain when it would survive process exit.

## 3. Cache and simulation are not live prices

The trading exercises use market data and cached values. A fixed fictional quote lets you observe caching. A fallback simulation must stay identifiable as simulated data.

**Predict:** How many provider-fixture calls occur for the same cached symbol?

In [ ]:
from functools import lru_cache
calls = []
@lru_cache(maxsize=8)
def practice_quote(symbol):
    calls.append(symbol)
    return {"symbol": symbol, "price": 100.0, "source": "fictional fixture"}
first = practice_quote("DEMO")
second = practice_quote("DEMO")
print(first, "Fixture calls:", len(calls))
assert len(calls) == 1 and second["source"] == "fictional fixture"

**Check:** One. Cached or random fallback prices are not verified current market values.

**Change one thing:** Call a second symbol. Explain how you would record a quote timestamp and expire old cache entries.

## 4. Provider endpoint and model selection

A compatible client can target different services. The endpoint selects the provider and its credentials; the model identifier selects a model there. Compatibility does not guarantee every feature works.

**Predict:** Does changing only the variable name change the provider?

In [ ]:
# Non-secret connection descriptions only; no clients or keys are created.
providers = {
    "openai": {"key_variable": "OPENAI_API_KEY", "endpoint": "default"},
    "gemini": {"key_variable": "GOOGLE_API_KEY", "endpoint": "https://generativelanguage.googleapis.com/v1beta/openai/"},
    "ollama": {"key_variable": None, "endpoint": "http://localhost:11434/v1"},
}
for provider, settings in providers.items():
    print(provider, settings["endpoint"], settings["key_variable"])
assert providers["gemini"]["key_variable"] == "GOOGLE_API_KEY"

**Check:** No. The configured endpoint/client and model select the service. Local Ollama still needs its service and downloaded model.

**Change one thing:** Add a provider from the linked source. Copy its endpoint and variable name, never a credential value.

## 5. Configuration stays outside executable examples

Environment-variable names identify settings. Values belong in .environment/.env and are loaded deliberately. This exercise inspects only the path and simulated presence flags; it reads no private file or variable values.

**Predict:** Does finding a file prove that authentication works?

In [ ]:
from pathlib import Path
# Find the repository from a kernel started inside it.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "1_foundations" / "3_lab3.ipynb").is_file()), None)
if root is None:
    raise FileNotFoundError("Start your kernel inside ai_agents")
configuration_path = root / ".environment" / ".env"
print("Configuration path:", configuration_path)
simulated_presence = {"OPENAI_API_KEY": True, "GOOGLE_API_KEY": False}
print(simulated_presence)  # Booleans only; these are fictional flags.
assert configuration_path.name == ".env"

**Check:** No. File existence, variable presence, and successful provider authentication are separate checks.

**Change one thing:** Explain the difference between a missing file, a missing variable, and a provider rejecting authentication. Do not print token prefixes or values.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.